In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Tue Jan 27 18:20:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd

In [9]:
df = pd.read_excel("/content/protein_constructs_w_label_masks.xlsx")[['rcsb_id', 'rcsb_entity_ids', 'uniprot_seq','pbd_id', 'pdb_sequence_sanitized', 'label_mask']]
df.head()

,rcsb_id,rcsb_entity_ids,uniprot_seq,pbd_id,pdb_sequence_sanitized,label_mask
0,2GD8,1,MSHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKP...,SHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPL...,SHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPL...,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,2GDD,1,MLNLLLLALPVLASRAYAAPAPGQALQRVGIVGGQEAPRSKWPWQV...,IVGGQEAPRSKWPWQVSLRVHGPYWMHFCGGSLIHPQWVLTAAHCV...,IVGGQEAPRSKWPWQVSLRVHGPYWMHFCGGSLIHPQWVLTAAHCV...,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,2GDE,1,MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRANT...,TFGSGEADCGLRPLFEKKSLEDKTERELLESYIDGR,TFGSGEADCGLRPLFEKKSLEDKTERELLESYIDGR,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,2GDE,2,MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRANT...,IVEGSDAEIGMSPWQVMLFRKSPQELLCGASLISDRWVLTAAHCLL...,IVEGSDAEIGMSPWQVMLFRKSPQELLCGASLISDRWVLTAAHCLL...,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,2GDO,1,MAVPFVEDWDLVQTLGEGAYGEVQLAVNRVTEEAVAVKIVDMKRAV...,MAVPFVEDWDLVQTLGEGAYGEVQLAVNRVTEEAVAVKIVDMKRAV...,MAVPFVEDWDLVQTLGEGAYGEVQLAVNRVTEEAVAVKIVDMKRAV...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [10]:
# model_name = "facebook/esm2_t33_650M_UR50D"

# tokenizer = AutoTokenizer.from_pretrained(model_name, do_lower_case=False)
# model = AutoModel.from_pretrained(model_name)

# model.eval()

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [12]:
from tqdm.auto import tqdm

In [15]:
# # import torch
# # from torch.utils.data import DataLoader, Dataset

# # class SeqDataset(Dataset):
# #     def __init__(self, sequences, tokenizer):
# #         self.sequences = sequences
# #         self.tokenizer = tokenizer

# #     def __len__(self):
# #         return len(self.sequences)

# #     def __getitem__(self, idx):
# #         seq = self.sequences[idx]
# #         tokens = self.tokenizer(seq, return_tensors="pt")
# #         # Remove batch dimension for collate_fn
# #         tokens = {k: v.squeeze(0) for k, v in tokens.items()}
# #         return tokens

# # batch_size = 4
# # dataset = SeqDataset(df['uniprot_seq'].tolist(), tokenizer)

# # def collate_fn(batch):
# #     batch = tokenizer.pad(
# #         batch,
# #         padding=True,
# #         return_tensors="pt"
# #     )
# #     return batch

# # dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_fn)

# # all_embeddings = []

# # model.eval()
# # model = model.to(device)

# # progress_bar = tqdm(dataloader, desc="Computing Embeddings", leave=True)

# # with torch.no_grad():
# #     for batch in progress_bar:
# #         batch = {k: v.to(device) for k, v in batch.items()}

# #         outputs = model(**batch)

# #         mask = batch['attention_mask'].unsqueeze(-1)
# #         summed = torch.sum(outputs.last_hidden_state * mask, dim=1)
# #         counts = torch.sum(mask, dim=1)
# #         seq_embeddings = summed / counts

# #         all_embeddings.append(seq_embeddings.cpu())

# # final_embeddings = torch.cat(all_embeddings, dim=0)
# # torch.save(final_embeddings, "pooled_embeddings.pt")
# # print("Final shape:", final_embeddings.shape)

# import torch
# from torch.utils.data import DataLoader, Dataset
# from tqdm.auto import tqdm

# # 1. Sort sequences by length (Dynamic Padding Optimization)
# # This ensures that each batch has sequences of similar length,
# # minimizing the number of 'pad' tokens and making it MUCH faster.
# df['seq_len'] = df['uniprot_seq'].str.len()
# df = df.sort_values('seq_len').reset_index(drop=True)

# # 2. Pre-tokenize the entire dataset (Removes the CPU bottleneck)
# print("Tokenizing sequences...")
# encodings = tokenizer(
#     df['uniprot_seq'].tolist(),
#     truncation=True,
#     padding=False, # We pad dynamically in the collate_fn
#     add_special_tokens=True
# )

# class FastSeqDataset(Dataset):
#     def __init__(self, encodings):
#         self.encodings = encodings

#     def __len__(self):
#         return len(self.encodings['input_ids'])

#     def __getitem__(self, idx):
#         return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}

# # 3. Setup Dataset and Optimized DataLoader
# dataset = FastSeqDataset(encodings)
# batch_size = 32  # Increased batch size for better GPU utilization

# def collate_fn(batch):
#     return tokenizer.pad(batch, padding=True, return_tensors="pt")

# dataloader = DataLoader(
#     dataset,
#     batch_size=batch_size,
#     collate_fn=collate_fn,
#     num_workers=4,       # Parallelize data preparation
#     pin_memory=True      # Faster data transfer to GPU
# )

# # 4. Model Preparation
# model.eval()
# model = model.to(device)

# # Optional: Use torch.compile for an extra 10-20% speed boost (PyTorch 2.0+)
# # if hasattr(torch, "compile"):
# #     model = torch.compile(model)

# all_embeddings = []

# # 5. Optimized Inference Loop
# with torch.no_grad():
#     # Use Automatic Mixed Precision (AMP) to speed up GPU math
#     with torch.cuda.amp.autocast():
#         for batch in tqdm(dataloader, desc="Generating Embeddings"):
#             # Move batch to GPU
#             batch = {k: v.to(device) for k, v in batch.items()}

#             outputs = model(**batch)

#             # Mean Pooling logic
#             mask = batch['attention_mask'].unsqueeze(-1)
#             summed = torch.sum(outputs.last_hidden_state * mask, dim=1)
#             counts = torch.sum(mask, dim=1)
#             seq_embeddings = summed / counts

#             # Move to CPU immediately to save GPU memory
#             all_embeddings.append(seq_embeddings.cpu())

# # 6. Finalize and Save
# final_embeddings = torch.cat(all_embeddings, dim=0)
# torch.save(final_embeddings, "pooled_embeddings.pt")
# print("Final shape:", final_embeddings.shape)

In [14]:
# --- CONFIGURATION ---
# Replace with your actual Hugging Face read token
# You can get one at https://huggingface.co/settings/tokens
import torch
from torch.utils.data import DataLoader, Dataset
hf_token = "***REMOVED***"

model_name = "ElnaggarLab/ankh-large"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import T5EncoderModel, AutoTokenizer

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    do_lower_case=False,
    token=hf_token
)

# Use T5EncoderModel specifically for Ankh/T5 protein models
model = T5EncoderModel.from_pretrained(
    model_name,
    token=hf_token
)
model.to(device).eval()


# 2. Sort sequences by length (Optimization for long proteins)
df['seq_len'] = df['uniprot_seq'].str.len()
df = df.sort_values('seq_len').reset_index(drop=True)

# 3. Pre-tokenize the entire dataset
print("Tokenizing sequences...")
# Ankh supports long context; capping at 2048 residues
encodings = tokenizer(
    df['uniprot_seq'].tolist(),
    truncation=True,
    max_length=2048,
    padding=False
)

class FastSeqDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}

# 4. Setup DataLoader
dataset = FastSeqDataset(encodings)
batch_size = 16 # Reduce to 4 or 8 if you run out of GPU memory (OOM)

def collate_fn(batch):
    return tokenizer.pad(batch, padding=True, return_tensors="pt")

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

all_embeddings = []

# 5. Optimized Inference Loop
print("Starting embedding generation...")
with torch.no_grad():
    # Use autocast for FP16 speed boost
    with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
        for batch in tqdm(dataloader, desc="Ankh Generation"):
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)

            # Mean Pooling: (sum of hidden states / number of non-pad tokens)
            mask = batch['attention_mask'].unsqueeze(-1)
            summed = torch.sum(outputs.last_hidden_state * mask, dim=1)
            counts = torch.sum(mask, dim=1)
            seq_embeddings = summed / counts

            # Move to CPU to prevent GPU memory bloat
            all_embeddings.append(seq_embeddings.cpu())

# 6. Finalize and Save
final_embeddings = torch.cat(all_embeddings, dim=0)
torch.save(final_embeddings, "ankh_embeddings.pt")
print(f"Success! Final shape: {final_embeddings.shape}")

Loading ElnaggarLab/ankh-large...
Tokenizing sequences...
Starting embedding generation...


/tmp/ipython-input-3242659017.py:71: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__

Ankh Generation:   0%|          | 0/4421 [00:00<?, ?it/s]

You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Success! Final shape: torch.Size([70732, 1536])


In [15]:
df['embeddings'] = list(final_embeddings.numpy())

In [16]:
df.columns

Index(['rcsb_id', 'rcsb_entity_ids', 'uniprot_seq', 'pbd_id',
       'pdb_sequence_sanitized', 'label_mask', 'seq_len', 'embeddings'],
      dtype='object')

In [17]:
df.to_parquet('protein_data.parquet', index=False)

In [18]:
df.to_pickle('protein_data.pkl')